<a href="https://colab.research.google.com/github/HenryZumaeta/py4dl_EPC2026/blob/main/C04_Script01_InicializacionPesos_callbacks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Punto de partida de la sesion 4
# https://gist.github.com/robintux/5c5931238fccdee6f5674ba44a76ef40

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn
from sklearn.model_selection import train_test_split
from sklearn import metrics

# Redes Neuronales : Keras
from keras.models import Sequential
from keras.layers import Dense, Input

# Eliminamos los datos descargados
os.system("rm *.csv")
os.system("rm *.zip")

# Descargamos el dataset desde kaggle
!kaggle datasets download robintux/salud-mental-agotamiento-estudiantil

# Descomprimimos el archivo descargado
!unzip salud-mental-agotamiento-estudiantil.zip

# Cargamos el dataset en memoria ram
df = pd.read_csv("Salud_mental_agotamiento_estudiantil.csv")
df = df.drop("Unnamed: 0", axis = 1)
df.info()

# Para este analisis vamos a hacer la siguiente consideracion
# Vamos a considerar valores de la variable dependiente mayores o iguales a 0.1
df = df[df.dropout_risk >= 0.1].reset_index(drop = True)
df.info()


# === OPCIONAL ===
# Trabajemos con una muestra (10% del total) del dataset para acortar el tiempo de procesamiento
df = df.sample(frac=0.1).reset_index(drop = True)
df.info()
# === OPCIONAL ===

# Definimos las variables a considerar en el modelamiento
# X : variables independientes
# y : variable dependiente
X = df.iloc[:, :-1]
y = df.dropout_risk


# Debemos particionar el dataset en dos subconjuntos
# Subconjunto de entrenamiento
# Subconjunto de testeo
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.25)
print(f"   NUmero de Observaciones (Train) : {X_train.shape[0]:d}")
print(f"   NUmero de Observaciones (Train) : {X_test.shape[0]:d}")


def build_mlp_regression(input_dim):

 # Construccion de un perceptron multicapa para nuestro problema de regresion
  # Instancia de la clase Sequential
  ModelBase = Sequential()

  # Agregamos capas ocultas
  ModelBase.add(Dense(
      units = 512,
      activation = "relu",
      input_dim = input_dim,
      name = "primera_capa_oculta"
  ))

  # Agreguemos 7 capas ocultas
  ModelBase.add(Dense(
      units = 256,
      activation = "relu",
      name = "segunda_capa_oculta"
  ))

  ModelBase.add(Dense(
      units = 128,
      activation = "relu",
      name = "tercera_capa_oculta"
  ))

  ModelBase.add(Dense(
      units = 96,
      activation = "relu",
      name = "cuarta_capa_oculta"
  ))

  ModelBase.add(Dense(
      units = 64,
      activation = "relu",
      name = "quinta_capa_oculta"
  ))

  ModelBase.add(Dense(
      units = 48,
      activation = "relu",
      name = "sexta_capa_oculta"
  ))

  ModelBase.add(Dense(
      units = 16,
      activation = "relu",
      name = "septima_capa_oculta"
  ))

  ModelBase.add(Dense(
      units = 4,
      activation = "relu",
      name = "octava_capa_oculta"
  ))

  # Capa de Salida
  ModelBase.add(Dense(
      units = 1,
      # activation="relu", # mape 100.00%
      # activation="celu", # mape 115.39%
      activation="relu",
    #   Despues de probar con la funcion de activacion linear
    #   una buena idea seria probar con la funcion de activacion softplus
    #   Otra opcion : linear + np.log(y)
      name = "capa_de_salida"
  ))


  # Definicion de relu(x) = max{0,x}

  ## OUTUT
  return ModelBase

# ==============================================================================

model1 = build_mlp_regression(X_train.shape[1])

# Compilacion del modelo
model1.compile(
    loss = "mean_absolute_error",
    optimizer="adam",
    metrics = ["mean_absolute_percentage_error"]
)

# Procedimiento de ajuste : Se realiza con los datos escalados
HistoriaAjusteModelBase = model1.fit(X_train, y_train , epochs = 50)

# Calculemos pronosticos de la variable dependiente (para X_test) : Usando el modelo
# recien ajustado/entrenado y los datos de testeo de la variable independiente
y_forecast_base = model1.predict(X_test)

# 8. Métricas finales
mape_final = metrics.mean_absolute_percentage_error(y_test, y_forecast_base)
mae_final = metrics.mean_absolute_error(y_test, y_forecast_base)
rmse_final = np.sqrt(metrics.mean_squared_error(y_test, y_forecast_base))
r2_final = metrics.r2_score(y_test, y_forecast_base)

print(f"MAPE: {mape_final:.4f} ({mape_final*100:.2f}%)")
print(f"MAE: {mae_final:.4f}")
print(f"RMSE: {rmse_final:.4f}")
print(f"R²: {r2_final:.4f}")

Dataset URL: https://www.kaggle.com/datasets/robintux/salud-mental-agotamiento-estudiantil
License(s): apache-2.0
100% 130M/130M [00:01<00:00, 123MB/s]

Archive:  salud-mental-agotamiento-estudiantil.zip
  inflating: Salud_mental_agotamiento_estudiantil.csv  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 17 columns):
 #   Column                Non-Null Count    Dtype  
---  ------                --------------    -----  
 0   age                   1000000 non-null  int64  
 1   study_hours_per_day   1000000 non-null  float64
 2   exam_pressure         1000000 non-null  float64
 3   academic_performance  1000000 non-null  float64
 4   stress_level          1000000 non-null  float64
 5   anxiety_score         1000000 non-null  float64
 6   depression_score      1000000 non-null  float64
 7   sleep_hours           1000000 non-null  float64
 8   physical_activity     1000000 non-null  float64
 9   social_support        1000000 non-null  

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1678/1678 ━━━━━━━━━━━━━━━━━━━━ 13s 6ms/step - loss: 1.8448 - mean_absolute_percentage_error: 100.0000
Epoch 2/50
1678/1678 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 1.8448 - mean_absolute_percentage_error: 100.0000
Epoch 3/50
1678/1678 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 1.8448 - mean_absolute_percentage_error: 100.0000
Epoch 4/50
1678/1678 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 1.8448 - mean_absolute_percentage_error: 100.0000
Epoch 5/50
1678/1678 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 1.8448 - mean_absolute_percentage_error: 100.0000
Epoch 6/50
1678/1678 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 1.8448 - mean_absolute_percentage_error: 100.0000
Epoch 7/50
1678/1678 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 1.8448 - mean_absolute_percentage_error: 100.0000
Epoch 8/50
1678/1678 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 1.8448 - mean_absolute_percentage_error: 100.0000
Epoch 9/50
1678/1678 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 1.8448 - mean_absolute_percentage_error: 100.00

In [2]:
# LIsta de funciones  de activacion
import keras.activations
dir(keras.activations)
# Analizar matematicamente cada una de las funciones que provee el submodulo keras.activations

['__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 'celu',
 'deserialize',
 'elu',
 'exponential',
 'gelu',
 'get',
 'glu',
 'hard_shrink',
 'hard_sigmoid',
 'hard_silu',
 'hard_swish',
 'hard_tanh',
 'leaky_relu',
 'linear',
 'log_sigmoid',
 'log_softmax',
 'mish',
 'relu',
 'relu6',
 'selu',
 'serialize',
 'sigmoid',
 'silu',
 'soft_shrink',
 'softmax',
 'softplus',
 'softsign',
 'sparse_plus',
 'sparse_sigmoid',
 'sparsemax',
 'squareplus',
 'swish',
 'tanh',
 'tanh_shrink',
 'threshold']

In [3]:
# Documentacion de celu
help(keras.activations.celu)

Help on function celu in module keras.src.activations.activations:

celu(x, alpha=1.0)
    Continuously Differentiable Exponential Linear Unit.

    The CeLU activation function is defined as:

    `celu(x) = alpha * (exp(x / alpha) - 1) for x < 0`,`celu(x) = x for x >= 0`.

    where `alpha` is a scaling parameter that controls the activation's shape.

    Args:
        x: Input tensor.
        alpha: The α value for the CeLU formulation. Defaults to `1.0`.

    Reference:

    - [Barron, J. T., 2017](https://arxiv.org/abs/1704.07483)



In [5]:
model1.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ primera_capa_oculta (Dense)     │ (None, 512)            │         8,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ segunda_capa_oculta (Dense)     │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ tercera_capa_oculta (Dense)     │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cuarta_capa_oculta (Dense)      │ (None, 96)             │        12,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ quinta_capa_oculta (Dense)      │ (None, 64)             │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sexta_capa_oculta (Dense)       │ (None, 48)             │         3,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ septima_capa_oculta (Dense)     │ (None, 16)             │           784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ octava_capa_oculta (Dense)      │ (None, 4)              │            68 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ capa_de_salida (Dense)          │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 586,493 (2.24 MB)

 Trainable params: 195,497 (763.66 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 390,996 (1.49 MB)

In [ ]:
# Explicabilidad del modelo (Interpretabilidad)
    # Explicación Estructural (¿Qué tiene el modelo?) -> el método summary
    # Explicación del Rendimiento (¿Qué tan bien aprendió?) -> El objeto/variable HistoriaAjusteModelBase
    # Explicación de las predicciones (XAI - Explaible AI) -> SHAP (Shapley Addditive Explanations). En python existe el módulo shap

# Inicializacion de Pesos

La inicializacion correcta es critica para :

* Convergencia de entrenamiento
* Estabilidad Numérica
* Reproducibilidad

In [6]:
import keras.initializers
dir(keras.initializers)

['Constant',
 'GlorotNormal',
 'GlorotUniform',
 'HeNormal',
 'HeUniform',
 'Identity',
 'IdentityInitializer',
 'Initializer',
 'LecunNormal',
 'LecunUniform',
 'Ones',
 'Orthogonal',
 'OrthogonalInitializer',
 'RandomNormal',
 'RandomUniform',
 'STFT',
 'STFTInitializer',
 'TruncatedNormal',
 'VarianceScaling',
 'Zeros',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 'constant',
 'deserialize',
 'get',
 'glorot_normal',
 'glorot_uniform',
 'he_normal',
 'he_uniform',
 'identity',
 'lecun_normal',
 'lecun_uniform',
 'ones',
 'orthogonal',
 'random_normal',
 'random_uniform',
 'serialize',
 'stft',
 'truncated_normal',
 'variance_scaling',
 'zeros']

In [7]:
# Veamos la documentacion de la clase Dense
help(Dense)

Help on class Dense in module keras.src.layers.core.dense:

class Dense(keras.src.layers.layer.Layer)
 |  Dense(units, activation=None, use_bias=True, kernel_initializer='glorot_uniform', bias_initializer='zeros', kernel_regularizer=None, bias_regularizer=None, activity_regularizer=None, kernel_constraint=None, bias_constraint=None, lora_rank=None, lora_alpha=None, quantization_config=None, **kwargs)
 |
 |  Just your regular densely-connected NN layer.
 |
 |  `Dense` implements the operation:
 |  `output = activation(dot(input, kernel) + bias)`
 |  where `activation` is the element-wise activation function
 |  passed as the `activation` argument, `kernel` is a weights matrix
 |  created by the layer, and `bias` is a bias vector created by the layer
 |  (only applicable if `use_bias` is `True`). When this layer is
 |  followed by a `BatchNormalization` layer, it is recommended to set
 |  `use_bias=False` as `BatchNormalization` has its own bias term.
 |
 |  Note: If the input to the lay

In [8]:
# Documentacion de glorot_uniform
help(keras.initializers.glorot_uniform)

Help on class GlorotUniform in module keras.src.initializers.random_initializers:

class GlorotUniform(VarianceScaling)
 |  GlorotUniform(seed=None)
 |
 |  The Glorot uniform initializer, also called Xavier uniform initializer.
 |
 |  Draws samples from a uniform distribution within `[-limit, limit]`, where
 |  `limit = sqrt(6 / (fan_in + fan_out))` (`fan_in` is the number of input
 |  units in the weight tensor and `fan_out` is the number of output units).
 |
 |  Examples:
 |
 |  >>> # Standalone usage:
 |  >>> initializer = GlorotUniform()
 |  >>> values = initializer(shape=(2, 2))
 |
 |  >>> # Usage in a Keras layer:
 |  >>> initializer = GlorotUniform()
 |  >>> layer = Dense(3, kernel_initializer=initializer)
 |
 |  Args:
 |      seed: A Python integer or instance of
 |          `keras.backend.SeedGenerator`.
 |          Used to make the behavior of the initializer
 |          deterministic. Note that an initializer seeded with an integer
 |          or `None` (unseeded) will produ

In [9]:
help(keras.initializers.RandomNormal)

Help on class RandomNormal in module keras.src.initializers.random_initializers:

class RandomNormal(RandomInitializer)
 |  RandomNormal(mean=0.0, stddev=0.05, seed=None)
 |
 |  Random normal initializer.
 |
 |  Draws samples from a normal distribution for given parameters.
 |
 |  Examples:
 |
 |  >>> # Standalone usage:
 |  >>> initializer = RandomNormal(mean=0.0, stddev=1.0)
 |  >>> values = initializer(shape=(2, 2))
 |
 |  >>> # Usage in a Keras layer:
 |  >>> initializer = RandomNormal(mean=0.0, stddev=1.0)
 |  >>> layer = Dense(3, kernel_initializer=initializer)
 |
 |  Args:
 |      mean: A python scalar or a scalar keras tensor. Mean of the random
 |          values to generate.
 |      stddev: A python scalar or a scalar keras tensor. Standard deviation of
 |         the random values to generate.
 |      seed: A Python integer or instance of
 |          `keras.backend.SeedGenerator`.
 |          Used to make the behavior of the initializer
 |          deterministic. Note that a

In [10]:
help(keras.initializers.Orthogonal)

Help on class Orthogonal in module keras.src.initializers.random_initializers:

class Orthogonal(RandomInitializer)
 |  Orthogonal(gain=1.0, seed=None)
 |
 |  Initializer that generates an orthogonal matrix.
 |
 |  If the shape of the tensor to initialize is two-dimensional, it is
 |  initialized with an orthogonal matrix obtained from the QR decomposition of
 |  a matrix of random numbers drawn from a normal distribution. If the matrix
 |  has fewer rows than columns then the output will have orthogonal rows.
 |  Otherwise, the output will have orthogonal columns.
 |
 |  If the shape of the tensor to initialize is more than two-dimensional,
 |  a matrix of shape `(shape[0] * ... * shape[n - 2], shape[n - 1])`
 |  is initialized, where `n` is the length of the shape vector.
 |  The matrix is subsequently reshaped to give a tensor of the desired shape.
 |
 |  Examples:
 |
 |  >>> # Standalone usage:
 |  >>> initializer = keras.initializers.Orthogonal()
 |  >>> values = initializer(shape

In [11]:
help(keras.initializers.STFT)

Help on class STFT in module keras.src.initializers.constant_initializers:

class STFT(keras.src.initializers.initializer.Initializer)
 |  STFT(side='real', window='hann', scaling='density', periodic=False)
 |
 |  Initializer of Conv kernels for Short-term Fourier Transformation (STFT).
 |
 |  Since the formula involves complex numbers, this class compute either the
 |  real or the imaginary components of the final output.
 |
 |  Additionally, this initializer supports windowing functions across the time
 |  dimension as commonly used in STFT. Windowing functions from the module
 |  `scipy.signal.windows` are supported, including the common `hann` and
 |  `hamming` windowing functions. This layer supports periodic windows and
 |  scaling-based normalization.
 |
 |  This is primarily intended for use in the `STFTSpectrogram` layer.
 |
 |  Examples:
 |
 |  >>> # Standalone usage:
 |  >>> initializer = STFTInitializer("real", "hann", "density", False)
 |  >>> values = initializer(shape=(1

# Mejora de la implementacion de : `build_mlp_regression`

In [12]:
# Implementación de una nueva versión de build_mlp_regression
def build_mlp_regression(
        input_dim,
        kernel_initializer = "glorot_uniform",
        bias_initializer = "zeros"):

    """
    Construye un MLP para regresion, permitiendo configurar la inicializacion.
    MLP: Multilayer Perceptron
    """

    # Instancia de la clase Sequential
    ModelBase = Sequential()

    # Lista de diccionarios con las configuraciones de capas para evitar repetir el código
    hidden_layers_config = [
        {"units" : 512 , "name" : "primera_capa_oculta"},
        {"units" : 256 , "name" : "segunda_capa_oculta"},
        {"units" : 128 , "name" : "tercera_capa_oculta"},
        {"units" : 96 , "name" : "cuarta_capa_oculta"},
        {"units" : 64 , "name" : "quinta_capa_oculta"},
        {"units" : 48 , "name" : "sexta_capa_oculta"},
        {"units" : 16 , "name" : "septima_capa_oculta"},
        {"units" : 4 , "name" : "occtava_capa_oculta"},
    ]

    # Agreguemos las capas ocultas en un bucle
    for i , config in enumerate(hidden_layers_config):
        ModelBase.add(Dense(
            units = config["units"],
            activation = "relu",
            input_dim = input_dim if i == 0 else None, # solo la primera capa oculta considera input_dim
            name = config["name"],
            kernel_initializer = kernel_initializer,
            bias_initializer = bias_initializer
        ))

    # Capa de salida
    ModelBase.add(Dense(
        units = 1,
        activation = "linear",
        name = "capa_de_salida",
        kernel_initializer = kernel_initializer,
        bias_initializer = bias_initializer
    ))

    return ModelBase


# Repliquemos model1
model1 = build_mlp_regression(X_train.shape[1])

# Compilacion del modelo
model1.compile(
    loss = "mean_absolute_error",
    optimizer="adam",
    metrics = ["mean_absolute_percentage_error"]
)

# Procedimiento de ajuste: Se realiza con los datos escalados
HistoriaAjusteModelBase = model1.fit(X_train, y_train , epochs = 10, batch_size=  16) # batch_size: potencia de dos.

# Calculemos pronosticos de la variable dependiente (para X_test) : Usando el modelo
# recien ajustado/entrenado y los datos de testeo de la variable independiente
y_forecast_base = model1.predict(X_test)

# Métricas finales
mape_final = metrics.mean_absolute_percentage_error(y_test, y_forecast_base)
mae_final = metrics.mean_absolute_error(y_test, y_forecast_base)
rmse_final = np.sqrt(metrics.mean_squared_error(y_test, y_forecast_base))
r2_final = metrics.r2_score(y_test, y_forecast_base)

print(f"MAPE: {mape_final:.4f} ({mape_final*100:.2f}%)")
print(f"MAE: {mae_final:.4f}")
print(f"RMSE: {rmse_final:.4f}")
print(f"R²: {r2_final:.4f}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 23s 5ms/step - loss: 0.8766 - mean_absolute_percentage_error: 76.5129
Epoch 2/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 20s 6ms/step - loss: 0.6778 - mean_absolute_percentage_error: 74.1611
Epoch 3/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 18s 5ms/step - loss: 0.6743 - mean_absolute_percentage_error: 74.2755
Epoch 4/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 18s 5ms/step - loss: 0.6724 - mean_absolute_percentage_error: 73.9458
Epoch 5/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 18s 5ms/step - loss: 0.6713 - mean_absolute_percentage_error: 73.7652
Epoch 6/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 18s 5ms/step - loss: 0.6724 - mean_absolute_percentage_error: 74.0708
Epoch 7/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 19s 6ms/step - loss: 0.6691 - mean_absolute_percentage_error: 73.2798
Epoch 8/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 18s 5ms/step - loss: 0.6702 - mean_absolute_percentage_error: 73.7406
Epoch 9/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 28s 8ms/step - loss: 0.6710 - mean_absolute_percentage_error: 

In [13]:
# Otra forma de implementacion de la funcion : build_mlp_regression
def build_mlp_regression(
        input_dim,
        hidden_units = [512,128,96,64,48,16,4],
        kernel_initializer = "he_normal",
        bias_initializer = "zeros"):
    ModelBase = Sequential()
    for i, units in enumerate(hidden_units):
        ModelBase.add(Dense(
            units = units,
            activation = "relu",
            input_dim = input_dim if i == 0 else None,
            name = f"capa_oculta_{i+1}",
            kernel_initializer = kernel_initializer,
            bias_initializer = bias_initializer
        ))
    ModelBase.add(Dense(
        units = 1 ,
        activation ="linear",
        name = "Capa_de_salida",
        kernel_initializer = kernel_initializer,
        bias_initializer = bias_initializer
    ))

    return ModelBase

# Experimentacion con diferentes arquitecturas
modelo_a = build_mlp_regression(input_dim=X_train.shape[1], hidden_units = [512,256,128])
modelo_b = build_mlp_regression(input_dim=X_train.shape[1], hidden_units = [16,24,32,48])
modelo_c = build_mlp_regression(input_dim=X_train.shape[1], hidden_units = [256,256,256])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [14]:
modelo_a.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ capa_oculta_1 (Dense)           │ (None, 512)            │         8,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ capa_oculta_2 (Dense)           │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ capa_oculta_3 (Dense)           │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Capa_de_salida (Dense)          │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 173,057 (676.00 KB)

 Trainable params: 173,057 (676.00 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
modelo_b.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ capa_oculta_1 (Dense)           │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ capa_oculta_2 (Dense)           │ (None, 24)             │           408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ capa_oculta_3 (Dense)           │ (None, 32)             │           800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ capa_oculta_4 (Dense)           │ (None, 48)             │         1,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Capa_de_salida (Dense)          │ (None, 1)              │            49 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,113 (12.16 KB)

 Trainable params: 3,113 (12.16 KB)

 Non-trainable params: 0 (0.00 B)

In [16]:
modelo_c.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ capa_oculta_1 (Dense)           │ (None, 256)            │         4,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ capa_oculta_2 (Dense)           │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ capa_oculta_3 (Dense)           │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Capa_de_salida (Dense)          │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 136,193 (532.00 KB)

 Trainable params: 136,193 (532.00 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# cada uno de estos modelos se deben de compilar , ajustar (metodo fit) , pronosticar (metodo predict) y calcular
# indicadores de calidad

In [17]:
# Compilacion del modelo
modelo_a.compile(
    loss = "mean_absolute_error",
    optimizer="adam",
    metrics = ["mean_absolute_percentage_error"]
)

# Procedimiento de ajuste: Se realiza con los datos escalados
HistoriaAjusteModelBase = modelo_a.fit(X_train, y_train , epochs = 10, batch_size=  16) # batch_size: potencia de dos.

# Calculemos pronosticos de la variable dependiente (para X_test) : Usando el modelo
# recien ajustado/entrenado y los datos de testeo de la variable independiente
y_forecast_base = modelo_a.predict(X_test)

# Métricas finales
mape_final = metrics.mean_absolute_percentage_error(y_test, y_forecast_base)
mae_final = metrics.mean_absolute_error(y_test, y_forecast_base)
rmse_final = np.sqrt(metrics.mean_squared_error(y_test, y_forecast_base))
r2_final = metrics.r2_score(y_test, y_forecast_base)

print(f"MAPE: {mape_final:.4f} ({mape_final*100:.2f}%)")
print(f"MAE: {mae_final:.4f}")
print(f"RMSE: {rmse_final:.4f}")
print(f"R²: {r2_final:.4f}")

Epoch 1/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 32s 9ms/step - loss: 0.8813 - mean_absolute_percentage_error: 97.7718
Epoch 2/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 28s 8ms/step - loss: 0.6965 - mean_absolute_percentage_error: 75.6420
Epoch 3/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 29s 5ms/step - loss: 0.6823 - mean_absolute_percentage_error: 74.4017
Epoch 4/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - loss: 0.6825 - mean_absolute_percentage_error: 74.3181
Epoch 5/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 17s 5ms/step - loss: 0.6761 - mean_absolute_percentage_error: 73.8589
Epoch 6/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 18s 4ms/step - loss: 0.6737 - mean_absolute_percentage_error: 73.5141
Epoch 7/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - loss: 0.6713 - mean_absolute_percentage_error: 73.6507
Epoch 8/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 17s 5ms/step - loss: 0.6704 - mean_absolute_percentage_error: 73.2722
Epoch 9/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 15s 5ms/step - loss: 0.6720 - mean_absolute_percentage_error: 

In [18]:
# Compilacion del modelo
modelo_b.compile(
    loss = "mean_absolute_error",
    optimizer="adam",
    metrics = ["mean_absolute_percentage_error"]
)

# Procedimiento de ajuste: Se realiza con los datos escalados
HistoriaAjusteModelBase = modelo_b.fit(X_train, y_train , epochs = 10, batch_size=  16) # batch_size: potencia de dos.

# Calculemos pronosticos de la variable dependiente (para X_test) : Usando el modelo
# recien ajustado/entrenado y los datos de testeo de la variable independiente
y_forecast_base = modelo_b.predict(X_test)

# Métricas finales
mape_final = metrics.mean_absolute_percentage_error(y_test, y_forecast_base)
mae_final = metrics.mean_absolute_error(y_test, y_forecast_base)
rmse_final = np.sqrt(metrics.mean_squared_error(y_test, y_forecast_base))
r2_final = metrics.r2_score(y_test, y_forecast_base)

print(f"MAPE: {mape_final:.4f} ({mape_final*100:.2f}%)")
print(f"MAE: {mae_final:.4f}")
print(f"RMSE: {rmse_final:.4f}")
print(f"R²: {r2_final:.4f}")

Epoch 1/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - loss: 0.7898 - mean_absolute_percentage_error: 88.2698 
Epoch 2/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.7019 - mean_absolute_percentage_error: 77.5214
Epoch 3/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - loss: 0.6899 - mean_absolute_percentage_error: 75.6907
Epoch 4/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.6807 - mean_absolute_percentage_error: 74.6719
Epoch 5/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 0.6765 - mean_absolute_percentage_error: 73.8344
Epoch 6/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.6734 - mean_absolute_percentage_error: 73.5195
Epoch 7/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - loss: 0.6723 - mean_absolute_percentage_error: 73.6306
Epoch 8/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.6703 - mean_absolute_percentage_error: 73.6016
Epoch 9/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - loss: 0.6701 - mean_absolute_percentage_error: 73.4795


In [19]:
# Compilacion del modelo
modelo_c.compile(
    loss = "mean_absolute_error",
    optimizer="adam",
    metrics = ["mean_absolute_percentage_error"]
)

# Procedimiento de ajuste: Se realiza con los datos escalados
HistoriaAjusteModelBase = modelo_c.fit(X_train, y_train , epochs = 10, batch_size=  16) # batch_size: potencia de dos.

# Calculemos pronosticos de la variable dependiente (para X_test) : Usando el modelo
# recien ajustado/entrenado y los datos de testeo de la variable independiente
y_forecast_base = modelo_c.predict(X_test)

# Métricas finales
mape_final = metrics.mean_absolute_percentage_error(y_test, y_forecast_base)
mae_final = metrics.mean_absolute_error(y_test, y_forecast_base)
rmse_final = np.sqrt(metrics.mean_squared_error(y_test, y_forecast_base))
r2_final = metrics.r2_score(y_test, y_forecast_base)

print(f"MAPE: {mape_final:.4f} ({mape_final*100:.2f}%)")
print(f"MAE: {mae_final:.4f}")
print(f"RMSE: {rmse_final:.4f}")
print(f"R²: {r2_final:.4f}")

Epoch 1/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - loss: 0.8476 - mean_absolute_percentage_error: 94.1092
Epoch 2/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - loss: 0.6938 - mean_absolute_percentage_error: 74.7923
Epoch 3/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - loss: 0.6837 - mean_absolute_percentage_error: 74.4320
Epoch 4/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - loss: 0.6790 - mean_absolute_percentage_error: 74.3438
Epoch 5/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - loss: 0.6766 - mean_absolute_percentage_error: 73.8878
Epoch 6/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - loss: 0.6745 - mean_absolute_percentage_error: 73.7207
Epoch 7/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - loss: 0.6740 - mean_absolute_percentage_error: 73.8135
Epoch 8/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - loss: 0.6721 - mean_absolute_percentage_error: 73.8408
Epoch 9/10
3355/3355 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - loss: 0.6717 - mean_absolute_percentage_error: 

# El submodulo `keras.callbacks`



In [20]:
# Veamos que provee el submodulo callbaks
import keras.callbacks
dir(keras.callbacks)

['BackupAndRestore',
 'CSVLogger',
 'Callback',
 'CallbackList',
 'EarlyStopping',
 'History',
 'LambdaCallback',
 'LearningRateScheduler',
 'ModelCheckpoint',
 'ProgbarLogger',
 'ReduceLROnPlateau',
 'RemoteMonitor',
 'SwapEMAWeights',
 'TensorBoard',
 'TerminateOnNaN',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__']

In [22]:
# Módulos
from keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler

# Semilla de reproducibilidad
np.random.seed(666) # Semilla en numpy
import tensorflow as tf
tf.random.set_seed(666) # Semilla en Tensorflow

# Fijación de la semilla en la parte cálculo vectorial y la parte tensorial.

In [23]:
# Escalar las variables independientes
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [24]:
# Listemos diferentes configuraciones de arquitecturas a probar
configuraciones = {
    # "Muy pequeña" : [16,8],
    # "Pequeña"     : [64,32, 16],
    # "Mediana"     : [128, 64, 32],
    "Grande"      : [256,128, 64],
    "Muy Grande"  : [512, 256, 128],
    # "Profunda"    : [256,256,256,256],
    # "Ancha y Profunda" : [1024, 512, 256, 128, 64]
}

# Creamos una lista en blanco para almacenar todos estos resultados
resultados = []

# Entrenar y evaluar cada configuracion
for nombre, hidden_units in configuraciones.items():
    print(f"--- Entrenando : {nombre} | Capas : {hidden_units}")

    # Construccion del modelo
    model = build_mlp_regression(
        X_train.shape[1],
        hidden_units = hidden_units
    )

    # Compilamos el modelo
    model.compile(
        loss = "mean_absolute_error",
        optimizer = "adam",
        metrics = ["mean_absolute_percentage_error"]
    )

    # Definamos la estrategia a utilizar en el callback (es un argumento del metodo fit)
    early_stop1 = EarlyStopping(
        monitor = "val_loss", # Vamos a considerar en el metodo fit un subconjunto (de X_train)  de validacion
        patience = 10, # La tolerancia para detenerse
        restore_best_weights = True, # Recuperar los mejores pesos después de detenerse
        verbose = 1 # Mostrar todo en consola
    )

    # Entrenamos el modelo
    history = model.fit(
        X_train_scaled, y_train,
        validation_split = 0.2, # 20% Para train
        epochs = 50,
        batch_size = 128,
        callbacks = [early_stop1], # early stoping
        verbose = 1 # Ver el proceso
    )

    # Prediccion sobre X_test_scaled
    y_pred = model.predict(X_test_scaled).flatten() # Se linealiza por el transform

    # Calculamos metricas
    mae = metrics.mean_absolute_error(y_test, y_pred)
    mape = metrics.mean_absolute_percentage_error(y_test, y_pred)
    rmse = np.sqrt(metrics.mean_squared_error(y_test, y_pred))
    r2 = metrics.r2_score(y_test, y_pred)

    # Guardamos resultados
    resultados.append({
        "Configuracion" : nombre,
        "Arquitectura"  : str(hidden_units),
        "MAE" : mae,
        "MAPE" : mape,
        "RMSE" : rmse,
        "R2" : r2,
        "Epocas Entrenadas" : len(history.history["loss"])
    })

    # Mostremos avances
    print(f" MAE : {mae:.4f} | MAPE : {mape:.4f} | {r2 :.4f} | Epocas : {len(history.history["loss"])}")

--- Entrenando : Grande | Capas : [256, 128, 64]


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
336/336 ━━━━━━━━━━━━━━━━━━━━ 9s 18ms/step - loss: 0.7200 - mean_absolute_percentage_error: 76.6508 - val_loss: 0.6896 - val_mean_absolute_percentage_error: 70.7525
Epoch 2/50
336/336 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.6806 - mean_absolute_percentage_error: 73.2844 - val_loss: 0.6812 - val_mean_absolute_percentage_error: 72.0140
Epoch 3/50
336/336 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.6697 - mean_absolute_percentage_error: 72.8400 - val_loss: 0.6771 - val_mean_absolute_percentage_error: 72.8045
Epoch 4/50
336/336 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.6643 - mean_absolute_percentage_error: 72.2572 - val_loss: 0.6735 - val_mean_absolute_percentage_error: 73.7634
Epoch 5/50
336/336 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.6613 - mean_absolute_percentage_error: 72.1958 - val_loss: 0.6735 - val_mean_absolute_percentage_error: 74.8245
Epoch 6/50
336/336 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 0.6576 - mean_absolute_percentage_error: 71.7747 - val_loss: 0.6759 

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


336/336 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 0.7312 - mean_absolute_percentage_error: 77.7817 - val_loss: 0.7013 - val_mean_absolute_percentage_error: 72.6036
Epoch 2/50
336/336 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.6867 - mean_absolute_percentage_error: 73.6829 - val_loss: 0.6839 - val_mean_absolute_percentage_error: 76.5193
Epoch 3/50
336/336 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.6752 - mean_absolute_percentage_error: 72.8316 - val_loss: 0.6792 - val_mean_absolute_percentage_error: 74.5439
Epoch 4/50
336/336 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 0.6685 - mean_absolute_percentage_error: 72.6537 - val_loss: 0.6768 - val_mean_absolute_percentage_error: 73.2302
Epoch 5/50
336/336 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 0.6646 - mean_absolute_percentage_error: 72.3557 - val_loss: 0.6755 - val_mean_absolute_percentage_error: 73.3971
Epoch 6/50
336/336 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 0.6598 - mean_absolute_percentage_error: 72.2176 - val_loss: 0.6752 - val_mean_

In [25]:
pd.DataFrame(resultados).sort_values(by = "MAPE")

,Configuracion,Arquitectura,MAE,MAPE,RMSE,R2,Epocas Entrenadas
1,Muy Grande,"[512, 256, 128]",0.675724,0.742199,0.860541,0.523913,16
0,Grande,"[256, 128, 64]",0.675573,0.743355,0.858996,0.525621,15
